# TF-IDF + Logistic Regression Model Evaluation

**Run cells top-to-bottom once** (e.g. *Run All* after a fresh kernel). The second code cell installs a pinned **`onnx>=1.14,<1.16`** for Intel Neural Compressor and `skl2onnx` compatibility, *before* later cells import `onnx`, so you should **not** need to restart the kernel between sections.

Evaluate the TF-IDF + Logistic Regression transaction classifier across serving strategies.

**ONNX / `skl2onnx`:** Character n-grams (`analyzer='char'` / `'char_wb'`) and custom analyzers are **not** convertible. For those cases, this notebook **automatically trains a word-level TF-IDF + LogisticRegression surrogate** on `data/transactions.csv` (payee → category) and exports **`models/tfidf_logreg_onnx/model.onnx`** from that pipeline so Triton and ORT benchmarks always have a valid ONNX file. Section 1 (sklearn baseline) still uses the **MLflow** model; sections 2–8 benchmark the **exported** ONNX (original pipeline when convertible, otherwise the surrogate—check printed messages).

Some strategies (GPU EPs, quantization) may have limited benefit for a linear model.

**Strategies** (2–8 run when ONNX export succeeds, including surrogate fallback):
1. Sklearn baseline (CPU) — always (MLflow model)
2. ONNX (no optimization) via skl2onnx
3. ONNX with graph optimization
4. Dynamic quantization
5. Static quantization - aggressive
6. Static quantization - conservative
7. GPU execution providers (CUDA, TensorRT)
8. OpenVINO execution provider

In [ ]:
import os
import sys


def find_repo_root():
    """Project root = directory containing scripts/ (works in Docker and bare VM)."""
    if os.environ.get("MLOPS_REPO"):
        return os.path.abspath(os.environ["MLOPS_REPO"])
    candidates = [
        "/home/jovyan/work",
        "/workspace",
        os.path.expanduser("~/mlops-project-proj06"),
        os.path.expanduser("~/serving-project"),
    ]
    p = os.getcwd()
    for _ in range(8):
        if p and os.path.isdir(os.path.join(p, "scripts")):
            candidates.append(p)
        p = os.path.dirname(p)
    for c in candidates:
        if c and os.path.isdir(os.path.join(c, "scripts")):
            return os.path.abspath(c)
    return os.path.abspath(os.getcwd())


REPO = find_repo_root()
os.chdir(REPO)
print(f"REPO={REPO}")

In [ ]:
# Pin onnx before skl2onnx / Neural Compressor import onnx.
# onnx>=1.14,<1.16: neural_compressor still uses onnx.mapping (removed in onnx 1.16+).
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "onnx>=1.14,<1.16",
        "skl2onnx",
        "ipywidgets",
    ]
)

In [ ]:
import os
import sys


def find_repo_root():
    """Project root = directory containing scripts/ (cwd may be evaluation/)."""
    if os.environ.get("MLOPS_REPO"):
        return os.path.abspath(os.environ["MLOPS_REPO"])
    candidates = [
        "/home/jovyan/work",
        "/workspace",
        os.path.expanduser("~/mlops-project-proj06"),
        os.path.expanduser("~/serving-project"),
    ]
    p = os.getcwd()
    for _ in range(8):
        if p and os.path.isdir(os.path.join(p, "scripts")):
            candidates.append(p)
        p = os.path.dirname(p)
    for c in candidates:
        if c and os.path.isdir(os.path.join(c, "scripts")):
            return os.path.abspath(c)
    return os.path.abspath(os.getcwd())


REPO = find_repo_root()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "scripts"))
print(f"REPO={REPO}")

import joblib
import numpy as np
import pandas as pd
from benchmark_utils import (
    get_model_size_mb, benchmark_latency, benchmark_batch_throughput,
    benchmark_ort_session, print_benchmark_results, collect_result_row
)

MODEL_NAME = "tfidf_logreg"
MODEL_DIR = os.path.join("models", MODEL_NAME)
ONNX_DIR = os.path.join("models", f"{MODEL_NAME}_onnx")
os.makedirs(ONNX_DIR, exist_ok=True)

RESULTS_DIR = os.path.join(REPO, "evaluation", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"MODEL_DIR={MODEL_DIR}")
print(f"ONNX_DIR={ONNX_DIR}")
print(f"RESULTS_DIR={RESULTS_DIR}")

# Set by ONNX export cell; ORT/quant cells skip when True.
SKIP_ONNX = False
ONNX_SKIP_REASON = None
# True when model.onnx comes from word-level surrogate (char/custom vectorizer).
ONNX_FALLBACK_USED = False

all_results = []

## Load sample data

In [ ]:
sample_texts = [
    "TESCO STORES 3456 -45.20 Friday",
    "DOMINOS #7548 -22.50 Thursday",
    "OLIVE GARDEN #9785 -67.30 Saturday",
    "AMAZON.COM -129.99 Monday",
    "SHELL OIL 12345 -55.00 Tuesday",
    "NETFLIX.COM -15.99 Wednesday",
    "WHOLE FOODS MKT -88.45 Sunday",
    "UBER *TRIP -18.75 Friday",
]

single_text = sample_texts[0]
batch_texts = sample_texts * 4  # 32 samples for batch benchmark

print(f"Single sample: '{single_text}'")
print(f"Batch size: {len(batch_texts)}")

## 1. Sklearn Baseline (CPU)

In [ ]:
# The model may be saved as .pkl, .joblib, or via MLflow format
# Try common patterns
model_file = None
for candidate in ['model.pkl', 'model.joblib', 'pipeline.pkl', 'pipeline.joblib', 'tfidf_logreg.joblib']:
    p = os.path.join(MODEL_DIR, candidate)
    if os.path.exists(p):
        model_file = p
        break

if model_file is None:
    mlflow_model_path = os.path.join(MODEL_DIR, 'model', 'model.pkl')
    if os.path.exists(mlflow_model_path):
        model_file = mlflow_model_path
    else:
        print(f"Model file not found in {MODEL_DIR}. Listing contents:")
        for root, dirs, files in os.walk(MODEL_DIR):
            for f in files:
                print(f"  {os.path.join(root, f)}")
        raise FileNotFoundError(f"Cannot find sklearn model in {MODEL_DIR}")

loaded = joblib.load(model_file)
if isinstance(loaded, dict) and "vectorizer" in loaded and "classifier" in loaded:
    from sklearn.pipeline import Pipeline
    pipeline = Pipeline([
        ("vectorizer", loaded["vectorizer"]),
        ("classifier", loaded["classifier"]),
    ])
else:
    pipeline = loaded

model_size = get_model_size_mb(MODEL_DIR)
print(f"Loaded model from {model_file}")
print(f"Model size: {model_size:.2f} MB")

In [ ]:
# Pipeline is built in the previous cell (dict bundles or a saved Pipeline).

In [ ]:
def predict_sklearn_single(text):
    return pipeline.predict_proba([text])

def predict_sklearn_batch(texts):
    return pipeline.predict_proba(texts)

latency_results = benchmark_latency(predict_sklearn_single, single_text)
batch_results = benchmark_batch_throughput(predict_sklearn_batch, batch_texts)
results_sklearn = {**latency_results, **batch_results}

print_benchmark_results(results_sklearn, MODEL_NAME, "sklearn_cpu", model_size)
all_results.append(collect_result_row(MODEL_NAME, "sklearn_cpu", "CPU", model_size, results_sklearn))

## 2. ONNX Export (No Optimization) via skl2onnx

If the MLflow pipeline cannot be converted, a **word-level surrogate** is trained on `data/transactions.csv` and written to `models/tfidf_logreg_onnx/model.onnx`. If both fail, sections **2–8** are skipped.

In [ ]:
import importlib.util
import os
import traceback

import onnxruntime as ort

# Load helper by file path (works even if sys.path does not include scripts/)
_helpers_py = os.path.join(REPO, "scripts", "tfidf_onnx_helpers.py")
if not os.path.isfile(_helpers_py):
    raise FileNotFoundError(
        f"Missing {_helpers_py}. Sync the repo so scripts/tfidf_onnx_helpers.py exists on this machine."
    )
_spec = importlib.util.spec_from_file_location("_tfidf_onnx_helpers", _helpers_py)
_tfh = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_tfh)
export_pipeline_to_onnx = _tfh.export_pipeline_to_onnx_file
write_surrogate_onnx = _tfh.write_surrogate_onnx

onnx_model_path = os.path.join(ONNX_DIR, "model.onnx")
SKIP_ONNX = False
ONNX_SKIP_REASON = None
ONNX_FALLBACK_USED = False


def _vectorizer_from_pipeline(pipe):
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

    for key in ("vectorizer", "tfidf", "tfidfvectorizer", "vect", "countvectorizer"):
        step = pipe.named_steps.get(key)
        if step is not None and isinstance(step, (TfidfVectorizer, CountVectorizer)):
            return step
    for _, step in pipe.named_steps.items():
        if isinstance(step, (TfidfVectorizer, CountVectorizer)):
            return step
    return None


def _pipeline_not_convertible_by_skl2onnx(pipe):
    """Return (skip, reason) if we should not attempt skl2onnx on this pipeline."""
    vec = _vectorizer_from_pipeline(pipe)
    if vec is None:
        return False, None
    an = getattr(vec, "analyzer", "word")
    if callable(an):
        return True, (
            "skl2onnx does not support custom analyzers; only word-level tokenizer is fully supported."
        )
    if an in ("char", "char_wb"):
        return True, (
            f"TfidfVectorizer/CountVectorizer with analyzer={an!r} cannot be converted by skl2onnx "
            "(only word-level is fully supported)."
        )
    return False, None


# --- Export: prefer MLflow pipeline; else surrogate from transactions.csv ---
onnx_size = None
not_conv, orig_reason = _pipeline_not_convertible_by_skl2onnx(pipeline)

if not_conv:
    print("Original pipeline is not directly convertible:", orig_reason)
    print("Fitting a word-level surrogate on data/transactions.csv for ONNX export only...")
    try:
        onnx_model_path = write_surrogate_onnx(REPO)
        ONNX_FALLBACK_USED = True
        onnx_size = get_model_size_mb(onnx_model_path)
        print(f"ONNX model (surrogate) exported to {onnx_model_path}")
        print(f"ONNX model size: {onnx_size:.2f} MB")
    except Exception as e:
        traceback.print_exc()
        SKIP_ONNX = True
        ONNX_SKIP_REASON = f"Fallback ONNX export failed: {e}"
        onnx_model_path = None
        onnx_size = None
else:
    try:
        if not os.path.exists(onnx_model_path):
            export_pipeline_to_onnx(pipeline, onnx_model_path)
            print(f"ONNX model exported to {onnx_model_path}")
        else:
            print(f"ONNX model already exists at {onnx_model_path}")
        onnx_size = get_model_size_mb(onnx_model_path)
        print(f"ONNX model size: {onnx_size:.2f} MB")
    except Exception as e:
        print(f"convert_sklearn failed on original pipeline: {e}")
        print("Retrying with word-level surrogate trained on data/transactions.csv...")
        try:
            onnx_model_path = write_surrogate_onnx(REPO)
            ONNX_FALLBACK_USED = True
            onnx_size = get_model_size_mb(onnx_model_path)
            print(f"ONNX model (surrogate) exported to {onnx_model_path}")
            print(f"ONNX model size: {onnx_size:.2f} MB")
        except Exception as e2:
            traceback.print_exc()
            SKIP_ONNX = True
            ONNX_SKIP_REASON = f"skl2onnx failed: {e}; fallback failed: {e2}"
            onnx_model_path = None
            onnx_size = None

# Safety net: empty tfidf_logreg_onnx/ is common if this cell errored or was skipped.
_onnx_target = os.path.join(ONNX_DIR, "model.onnx")
if not os.path.isfile(_onnx_target):
    print(
        f"No file at {_onnx_target}; training surrogate "
        f"(transactions.csv in data/, models/, or repo root)..."
    )
    try:
        onnx_model_path = write_surrogate_onnx(REPO)
        SKIP_ONNX = False
        ONNX_FALLBACK_USED = True
        onnx_size = get_model_size_mb(onnx_model_path)
        print(f"ONNX model (surrogate) written to {onnx_model_path} ({onnx_size:.2f} MB)")
    except Exception as e3:
        traceback.print_exc()
        SKIP_ONNX = True
        ONNX_SKIP_REASON = (ONNX_SKIP_REASON or " ") + f" Surrogate safety-net failed: {e3}"
        onnx_model_path = None
        onnx_size = None

if SKIP_ONNX:
    onnx_model_path = None
    onnx_size = None
    print("Skipping ONNX and all ONNX Runtime / quantization / GPU EP benchmarks below.")
    if ONNX_SKIP_REASON:
        print("Reason:", ONNX_SKIP_REASON)
elif ONNX_FALLBACK_USED:
    print(
        "\nNote: ONNX benchmarks (sections 2–8) use the word-level surrogate, not the MLflow sklearn baseline."
    )

In [ ]:
if not SKIP_ONNX:
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

    ort_session_no_opt = ort.InferenceSession(
        onnx_model_path,
        sess_options=session_options,
        providers=['CPUExecutionProvider']
    )

    print(f"Providers: {ort_session_no_opt.get_providers()}")

    input_name = ort_session_no_opt.get_inputs()[0].name
    print(f"ONNX input name: {input_name}")

    def make_ort_input(texts):
        if isinstance(texts, str):
            texts = [texts]
        return {input_name: np.array(texts).reshape(-1, 1)}

    sample_ort = make_ort_input(single_text)
    batch_ort = make_ort_input(batch_texts)

    results_onnx_no_opt = benchmark_ort_session(ort_session_no_opt, sample_ort, batch_ort)
    print_benchmark_results(results_onnx_no_opt, MODEL_NAME, "onnx_no_opt", onnx_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_no_opt", "CPU", onnx_size, results_onnx_no_opt))

## 3. ONNX with Graph Optimization (ORT_ENABLE_EXTENDED)

In [ ]:
if not SKIP_ONNX:
    optimized_model_path = os.path.join(ONNX_DIR, "model_optimized.onnx")

    session_options_opt = ort.SessionOptions()
    session_options_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED
    session_options_opt.optimized_model_filepath = optimized_model_path

    ort_session_opt = ort.InferenceSession(
        onnx_model_path,
        sess_options=session_options_opt,
        providers=['CPUExecutionProvider']
    )

    opt_size = get_model_size_mb(optimized_model_path) if os.path.exists(optimized_model_path) else onnx_size
    print(f"Optimized ONNX model size: {opt_size:.2f} MB")

    ort_session_opt_loaded = ort.InferenceSession(
        optimized_model_path if os.path.exists(optimized_model_path) else onnx_model_path,
        providers=['CPUExecutionProvider']
    )

    results_onnx_opt = benchmark_ort_session(ort_session_opt_loaded, sample_ort, batch_ort)
    print_benchmark_results(results_onnx_opt, MODEL_NAME, "onnx_optimized", opt_size)
    all_results.append(collect_result_row(MODEL_NAME, "onnx_optimized", "CPU", opt_size, results_onnx_opt))

## 4. Dynamic Quantization (Intel Neural Compressor)

In [ ]:
if not SKIP_ONNX:
    import neural_compressor
    from neural_compressor import quantization

    fp32_model = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

    config_dynamic = neural_compressor.PostTrainingQuantConfig(
        approach="dynamic"
    )

    q_model_dynamic = quantization.fit(
        model=fp32_model,
        conf=config_dynamic
    )

    dynamic_quant_path = os.path.join(ONNX_DIR, "model_quantized_dynamic.onnx")
    q_model_dynamic.save_model_to_file(dynamic_quant_path)

    dq_size = get_model_size_mb(dynamic_quant_path)
    print(f"Dynamic quantized model size: {dq_size:.2f} MB")

In [ ]:
if not SKIP_ONNX:
    ort_session_dq = ort.InferenceSession(dynamic_quant_path, providers=['CPUExecutionProvider'])

    results_dq = benchmark_ort_session(ort_session_dq, sample_ort, batch_ort)
    print_benchmark_results(results_dq, MODEL_NAME, "dynamic_quant", dq_size)
    all_results.append(collect_result_row(MODEL_NAME, "dynamic_quant", "CPU", dq_size, results_dq))

## 5. Static Quantization - Aggressive

Note: For a linear model (TF-IDF + LogReg), quantization benefit is typically minimal.
The model parameters are already simple linear weights. We measure it for completeness.

In [ ]:
if not SKIP_ONNX:
    import neural_compressor
    from neural_compressor import quantization
    from torch.utils.data import Dataset

    ort_input_names = [inp.name for inp in ort_session_no_opt.get_inputs()]
    print(f"ONNX inputs for calibration: {ort_input_names}")

    class TextCalibrationDataset(Dataset):
        """Calibration rows must match ONNX graph inputs (names and count)."""

        def __init__(self, csv_path, onnx_input_names, max_samples=256):
            df = pd.read_csv(csv_path)
            self.texts = df['payee'].dropna().unique()[:max_samples].tolist()
            self.onnx_input_names = list(onnx_input_names)

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            feed = {}
            for name in self.onnx_input_names:
                feed[name] = np.array([self.texts[idx]]).reshape(1, 1)
            return (feed, 0)

    calib_dataset = TextCalibrationDataset(
        os.path.join(REPO, "data", "transactions.csv"),
        ort_input_names,
    )
    calib_dataloader = neural_compressor.data.DataLoader(
        framework='onnxruntime',
        dataset=calib_dataset,
        batch_size=1
    )

    fp32_model_agg = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

    config_static_aggressive = neural_compressor.PostTrainingQuantConfig(
        accuracy_criterion=neural_compressor.config.AccuracyCriterion(
            criterion="absolute",
            tolerable_loss=0.05
        ),
        approach="static",
        device='cpu',
        quant_level=1,
        quant_format="QOperator",
        recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
        calibration_sampling_size=128
    )

    try:
        q_model_agg = quantization.fit(
            model=fp32_model_agg,
            conf=config_static_aggressive,
            calib_dataloader=calib_dataloader,
        )
        agg_quant_path = os.path.join(ONNX_DIR, "model_quantized_aggressive.onnx")
        q_model_agg.save_model_to_file(agg_quant_path)
        agg_size = get_model_size_mb(agg_quant_path)
        print(f"Aggressive quantized model size: {agg_size:.2f} MB")

        ort_session_agg = ort.InferenceSession(agg_quant_path, providers=['CPUExecutionProvider'])
        results_agg = benchmark_ort_session(ort_session_agg, sample_ort, batch_ort)
        print_benchmark_results(results_agg, MODEL_NAME, "static_quant_aggressive", agg_size)
        all_results.append(collect_result_row(MODEL_NAME, "static_quant_aggressive", "CPU", agg_size, results_agg))
    except Exception as e:
        print(f"Static quantization (aggressive) failed for TF-IDF+LogReg: {e}")
        print("This is expected for simple linear models with limited quantizable ops.")

## 6. Static Quantization - Conservative

In [ ]:
if not SKIP_ONNX:
    import neural_compressor
    from neural_compressor import quantization

    fp32_model_con = neural_compressor.model.onnx_model.ONNXModel(onnx_model_path)

    config_static_conservative = neural_compressor.PostTrainingQuantConfig(
        accuracy_criterion=neural_compressor.config.AccuracyCriterion(
            criterion="absolute",
            tolerable_loss=0.01
        ),
        approach="static",
        device='cpu',
        quant_level=0,
        quant_format="QOperator",
        recipes={"graph_optimization_level": "ENABLE_EXTENDED"},
        calibration_sampling_size=128
    )

    try:
        q_model_con = quantization.fit(
            model=fp32_model_con,
            conf=config_static_conservative,
            calib_dataloader=calib_dataloader,
        )
        con_quant_path = os.path.join(ONNX_DIR, "model_quantized_conservative.onnx")
        q_model_con.save_model_to_file(con_quant_path)
        con_size = get_model_size_mb(con_quant_path)
        print(f"Conservative quantized model size: {con_size:.2f} MB")

        ort_session_con = ort.InferenceSession(con_quant_path, providers=['CPUExecutionProvider'])
        results_con = benchmark_ort_session(ort_session_con, sample_ort, batch_ort)
        print_benchmark_results(results_con, MODEL_NAME, "static_quant_conservative", con_size)
        all_results.append(collect_result_row(MODEL_NAME, "static_quant_conservative", "CPU", con_size, results_con))
    except Exception as e:
        print(f"Static quantization (conservative) failed for TF-IDF+LogReg: {e}")
        print("This is expected for simple linear models with limited quantizable ops.")

## 7. GPU Execution Providers

**Note**: Run this section using the `jupyter-eval-gpu` container (Dockerfile.jupyter-eval-gpu) which has `onnxruntime-gpu` installed.

GPU acceleration is unlikely to benefit TF-IDF+LogReg due to the model's simplicity
and the overhead of CPU-to-GPU data transfer. We measure for completeness.

In [ ]:
if not SKIP_ONNX:
    try:
        ort_session_cuda = ort.InferenceSession(
            onnx_model_path,
            providers=['CUDAExecutionProvider']
        )
        print(f"CUDA providers: {ort_session_cuda.get_providers()}")

        results_cuda = benchmark_ort_session(ort_session_cuda, sample_ort, batch_ort)
        print_benchmark_results(results_cuda, MODEL_NAME, "onnx_cuda", onnx_size)
        all_results.append(collect_result_row(MODEL_NAME, "onnx_cuda", "P100", onnx_size, results_cuda))
    except Exception as e:
        print(f"CUDAExecutionProvider not available or not applicable: {e}")
        print("Run this cell in the GPU container (jupyter-eval-gpu).")

In [ ]:
if not SKIP_ONNX:
    try:
        ort_session_trt = ort.InferenceSession(
            onnx_model_path,
            providers=['TensorrtExecutionProvider']
        )
        print(f"TensorRT providers: {ort_session_trt.get_providers()}")

        results_trt = benchmark_ort_session(ort_session_trt, sample_ort, batch_ort)
        print_benchmark_results(results_trt, MODEL_NAME, "onnx_tensorrt", onnx_size)
        all_results.append(collect_result_row(MODEL_NAME, "onnx_tensorrt", "P100", onnx_size, results_trt))
    except Exception as e:
        print(f"TensorrtExecutionProvider not available or not applicable: {e}")
        print("Run this cell in the GPU container (jupyter-eval-gpu).")

## 8. OpenVINO Execution Provider

**Note**: Run this section using the `jupyter-eval-openvino` container (Dockerfile.jupyter-eval-openvino).

In [ ]:
if not SKIP_ONNX:
    try:
        ort_session_ov = ort.InferenceSession(
            onnx_model_path,
            providers=['OpenVINOExecutionProvider']
        )
        print(f"OpenVINO providers: {ort_session_ov.get_providers()}")

        results_ov = benchmark_ort_session(ort_session_ov, sample_ort, batch_ort)
        print_benchmark_results(results_ov, MODEL_NAME, "onnx_openvino", onnx_size)
        all_results.append(collect_result_row(MODEL_NAME, "onnx_openvino", "CPU (OpenVINO)", onnx_size, results_ov))
    except Exception as e:
        print(f"OpenVINOExecutionProvider not available or not applicable: {e}")
        print("Run this cell in the OpenVINO container (jupyter-eval-openvino).")

## Summary

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
out_csv = os.path.join(RESULTS_DIR, f"{MODEL_NAME}_evaluation.csv")
results_df.to_csv(out_csv, index=False)
print(f"\nResults saved to {out_csv}")
if SKIP_ONNX:
    print("\n(ONNX was skipped; CSV contains sklearn baseline only.)")
elif ONNX_FALLBACK_USED:
    print(
        "\n(ONNX file and ONNX strategy rows use the word-level surrogate; sklearn_cpu uses the MLflow model.)"
    )